# 16 — Leave-One-Image-Out RelMS Sweep

Notebook 10 found that `cor`, `gfc_sag`, `gfc_tra` are mutually redundant (pairwise
correlation 0.45-0.60 between their distance matrices), while `masked_tra` and `sbj_sag`
are the most independent / complementary views (r as low as 0.016). Notebook 08 found
that merging **all 5** images via RelMS gives weaker ARI than `masked_tra` used **alone**
(notebook 05).

This notebook tests that observation systematically instead of via ad-hoc pairs:
for each of the 5 image types, cluster on the RelMS distance built from the **other 4**
image types and compare against the full 5-image baseline.

**Logic:**
- If removing an image type **improves** ARI/NMI relative to the full 5-image baseline,
  that image was diluting the signal (its block disagrees with / adds noise relative to CDR).
- If removing an image type **hurts** ARI/NMI, that image was contributing real signal.

All 6 RelMS distance matrices (full + 5 leave-one-out) use equal block weighting
(`weights=None`), `d1='euclidean'`, no clinical variables — isolating the imaging
contribution only, same scope as notebook 08.

## Step 1 — Install / verify package version

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-600:] if r.stderr else "(no stderr)")

run_pip("robust-mixed-dist==0.1.26")
for pkg in ["kmedoids", "openpyxl"]:
    run_pip(pkg)
print("Done.")

import importlib.metadata
print(f"\nrobust-mixed-dist version: {importlib.metadata.version('robust-mixed-dist')}")

  OK    robust-mixed-dist==0.1.26
  OK    kmedoids
  OK    openpyxl
Done.

robust-mixed-dist version: 0.1.26


## Imports

In [2]:
import os
import numpy as np
import pandas as pd

import kmedoids
# function lives in robust_mixed_dist.mixed, not at the package top level
from robust_mixed_dist.mixed import related_metric_scaling_dist_matrix_faster
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 2 — Load the 5 PCA files

Each file is 416 x 111 (patient_id + 110 PCA components), produced by notebook 02.
Kept as separate blocks (not concatenated) so any subset can be assembled per experiment.

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

IMAGE_TYPES = ["cor", "gfc_sag", "gfc_tra", "masked_tra", "sbj_sag"]
PCA_FILES   = {t: f"pca_{t}.csv" for t in IMAGE_TYPES}

blocks      = {}   # image_type -> (416, 110) array
patient_ids = None

for itype, fname in PCA_FILES.items():
    df = pd.read_csv(os.path.join(NB_DIR, fname))
    if patient_ids is None:
        patient_ids = df["patient_id"].values
    else:
        assert np.array_equal(patient_ids, df["patient_id"].values), f"Patient order mismatch in {fname}"
    pca_cols = [c for c in df.columns if c != "patient_id"]
    blocks[itype] = df[pca_cols].to_numpy(dtype=np.float64)
    print(f"  {itype:<14} -> block shape {blocks[itype].shape}")

print(f"\nAll 5 blocks loaded and aligned -> {len(patient_ids)} patients.")

  cor            -> block shape (416, 110)
  gfc_sag        -> block shape (416, 110)
  gfc_tra        -> block shape (416, 110)
  masked_tra     -> block shape (416, 110)
  sbj_sag        -> block shape (416, 110)

All 5 blocks loaded and aligned -> 416 patients.


## Load CDR for evaluation

CDR is **not** used during clustering -- only for post-hoc evaluation.

In [4]:
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)

cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)

print(f"Clinical table    : {df_clin.shape}")
print(f"Patients with CDR : {int(mask_cdr.sum())} / {len(patient_ids)}")
print(f"CDR categories    : {sorted(set(cdr_int))}  (CDR x 2)")

Clinical table    : (416, 12)
Patients with CDR : 235 / 416
CDR categories    : [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]  (CDR x 2)


## Step 3 — Helper: build RelMS distance, cluster, evaluate

Given an ordered list of image-type blocks, concatenate them, compute the RelMS distance
matrix with one block per image type (`p1` = list of 110s, one per included image type),
then run FasterPAM for k=3 and k=4 and compute the same evaluation metrics used throughout
the project (spread of %healthy, silhouette, ARI, NMI).

In [5]:
def run_relms_experiment(included_types, label):
    X_blocks = [blocks[t] for t in included_types]
    X        = np.concatenate(X_blocks, axis=1)
    p1_list  = [b.shape[1] for b in X_blocks]

    print(f"\n{'='*62}")
    print(f"  {label}  |  images={included_types}  |  X={X.shape}")
    print(f"{'='*62}")

    D = related_metric_scaling_dist_matrix_faster(
        X=pd.DataFrame(X),   # DataFrame required -- avoids numpy bug in 0.1.26
        p1=p1_list,
        p2=0,
        p3=0,
        d1="euclidean",
        d2="sokal",
        d3="hamming",
        weights=None,
    )
    print(f"  D shape={D.shape}  range=[{D.min():.4f}, {D.max():.4f}]  "
          f"symmetric={np.allclose(D, D.T)}  NaN={np.isnan(D).sum()}")

    rows = []
    for k in [3, 4]:
        result = kmedoids.fasterpam(D, medoids=k, random_state=42)
        labels = np.array(result.labels, dtype=int)

        df_eval = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
        df_eval = df_eval[df_eval["CDR"].notnull()].copy()
        pct_tab = pd.crosstab(
            df_eval["cluster"], df_eval["CDR"], normalize="index"
        ).mul(100).round(1)
        pct_healthy = pct_tab[0.0] if 0.0 in pct_tab.columns \
                      else pd.Series(0.0, index=pct_tab.index)
        spread = float(pct_healthy.max() - pct_healthy.min())

        sil = silhouette_score(D, labels, metric="precomputed")
        ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
        nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

        sizes = {c: int((labels == c).sum()) for c in sorted(set(labels))}
        print(f"  k={k}  loss={result.loss:.4f}  sizes={sizes}")
        print(f"    spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

        rows.append({
            "experiment":         label,
            "images_used":        ",".join(included_types),
            "n_images":           len(included_types),
            "k":                  k,
            "silhouette":         round(sil, 4),
            "ari":                round(ari, 4),
            "nmi":                round(nmi, 4),
            "spread_pct_healthy": round(spread, 2),
        })
    return rows

print("Helper defined.")

Helper defined.


## Step 4 — Baseline: all 5 images (recomputes notebook 08's result for direct comparison)

In [6]:
all_results = []
all_results += run_relms_experiment(IMAGE_TYPES, "full_5_images")


  full_5_images  |  images=['cor', 'gfc_sag', 'gfc_tra', 'masked_tra', 'sbj_sag']  |  X=(416, 550)


  D shape=(416, 416)  range=[0.0000, 3.9054]  symmetric=True  NaN=0
  k=3  loss=1144.4943  sizes={np.int64(0): 135, np.int64(1): 115, np.int64(2): 166}
    spread=20.20%  sil=0.0183  ARI=-0.0103  NMI=0.0233
  k=4  loss=1133.5998  sizes={np.int64(0): 105, np.int64(1): 107, np.int64(2): 130, np.int64(3): 74}
    spread=18.90%  sil=0.0106  ARI=-0.0056  NMI=0.0240


## Step 5 — Leave-one-image-out sweep

For each image type, build the RelMS distance from the remaining 4 and cluster.

In [7]:
for leave_out in IMAGE_TYPES:
    remaining = [t for t in IMAGE_TYPES if t != leave_out]
    all_results += run_relms_experiment(remaining, f"leave_out_{leave_out}")

print("\n\nAll leave-one-out experiments complete.")


  leave_out_cor  |  images=['gfc_sag', 'gfc_tra', 'masked_tra', 'sbj_sag']  |  X=(416, 440)
  D shape=(416, 416)  range=[0.0000, 3.4716]  symmetric=True  NaN=0


  k=3  loss=1023.2553  sizes={np.int64(0): 157, np.int64(1): 144, np.int64(2): 115}
    spread=1.10%  sil=0.0224  ARI=-0.0084  NMI=0.0133


  k=4  loss=1012.6432  sizes={np.int64(0): 96, np.int64(1): 103, np.int64(2): 98, np.int64(3): 119}
    spread=2.90%  sil=0.0123  ARI=-0.0056  NMI=0.0085

  leave_out_gfc_sag  |  images=['cor', 'gfc_tra', 'masked_tra', 'sbj_sag']  |  X=(416, 440)
  D shape=(416, 416)  range=[0.0000, 3.5253]  symmetric=True  NaN=0
  k=3  loss=1021.0226  sizes={np.int64(0): 171, np.int64(1): 96, np.int64(2): 149}
    spread=5.40%  sil=0.0216  ARI=-0.0147  NMI=0.0084
  k=4  loss=1009.7228  sizes={np.int64(0): 89, np.int64(1): 86, np.int64(2): 123, np.int64(3): 118}
    spread=34.70%  sil=0.0213  ARI=-0.0112  NMI=0.0366

  leave_out_gfc_tra  |  images=['cor', 'gfc_sag', 'masked_tra', 'sbj_sag']  |  X=(416, 440)


  D shape=(416, 416)  range=[0.0000, 3.5850]  symmetric=True  NaN=0
  k=3  loss=1022.0790  sizes={np.int64(0): 181, np.int64(1): 108, np.int64(2): 127}
    spread=15.60%  sil=0.0248  ARI=-0.0144  NMI=0.0186
  k=4  loss=1010.4663  sizes={np.int64(0): 116, np.int64(1): 100, np.int64(2): 88, np.int64(3): 112}
    spread=10.40%  sil=0.0183  ARI=-0.0163  NMI=0.0267

  leave_out_masked_tra  |  images=['cor', 'gfc_sag', 'gfc_tra', 'sbj_sag']  |  X=(416, 440)
  D shape=(416, 416)  range=[0.0000, 3.5803]  symmetric=True  NaN=0
  k=3  loss=1002.2500  sizes={np.int64(0): 124, np.int64(1): 123, np.int64(2): 169}
    spread=8.30%  sil=0.0239  ARI=-0.0039  NMI=0.0171
  k=4  loss=992.8354  sizes={np.int64(0): 86, np.int64(1): 98, np.int64(2): 112, np.int64(3): 120}
    spread=34.50%  sil=0.0211  ARI=-0.0029  NMI=0.0402

  leave_out_sbj_sag  |  images=['cor', 'gfc_sag', 'gfc_tra', 'masked_tra']  |  X=(416, 440)


  D shape=(416, 416)  range=[0.0000, 3.5252]  symmetric=True  NaN=0
  k=3  loss=1024.8512  sizes={np.int64(0): 111, np.int64(1): 166, np.int64(2): 139}
    spread=20.20%  sil=0.0069  ARI=-0.0076  NMI=0.0243
  k=4  loss=1016.4824  sizes={np.int64(0): 93, np.int64(1): 122, np.int64(2): 98, np.int64(3): 103}
    spread=24.60%  sil=0.0035  ARI=-0.0078  NMI=0.0322


All leave-one-out experiments complete.


## Step 6 — Summary table: delta vs. full-5-image baseline

`delta_ari` / `delta_nmi` = (leave-one-out value) - (full 5-image value) at the same k.
**Positive delta = removing that image improved CDR alignment** (it was diluting the signal).
**Negative delta = removing that image hurt CDR alignment** (it was contributing real signal).

In [8]:
df_results = pd.DataFrame(all_results)

baseline = df_results[df_results["experiment"] == "full_5_images"].set_index("k")

def add_delta(row):
    if row["experiment"] == "full_5_images":
        return pd.Series({"delta_ari": 0.0, "delta_nmi": 0.0})
    base = baseline.loc[row["k"]]
    return pd.Series({
        "delta_ari": round(row["ari"] - base["ari"], 4),
        "delta_nmi": round(row["nmi"] - base["nmi"], 4),
    })

df_results[["delta_ari", "delta_nmi"]] = df_results.apply(add_delta, axis=1)

print("Leave-one-image-out RelMS sweep -- full results")
print("=" * 100)
print(df_results.to_string(index=False))

out_path = os.path.join(NB_PADDED, "results_leave_one_out.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Leave-one-image-out RelMS sweep -- full results
          experiment                            images_used  n_images  k  silhouette     ari    nmi  spread_pct_healthy  delta_ari  delta_nmi
       full_5_images cor,gfc_sag,gfc_tra,masked_tra,sbj_sag         5  3      0.0183 -0.0103 0.0233                20.2     0.0000     0.0000
       full_5_images cor,gfc_sag,gfc_tra,masked_tra,sbj_sag         5  4      0.0106 -0.0056 0.0240                18.9     0.0000     0.0000
       leave_out_cor     gfc_sag,gfc_tra,masked_tra,sbj_sag         4  3      0.0224 -0.0084 0.0133                 1.1     0.0019    -0.0100
       leave_out_cor     gfc_sag,gfc_tra,masked_tra,sbj_sag         4  4      0.0123 -0.0056 0.0085                 2.9     0.0000    -0.0155
   leave_out_gfc_sag         cor,gfc_tra,masked_tra,sbj_sag         4  3      0.0216 -0.0147 0.0084                 5.4    -0.0044    -0.0149
   leave_out_gfc_sag         cor,gfc_tra,masked_tra,sbj_sag         4  4      0.0213 -0.0112 0.0366 

## Step 7 — Interpretation: which images help vs. hurt?

In [9]:
loo = df_results[df_results["experiment"] != "full_5_images"].copy()
loo["left_out"] = loo["experiment"].str.replace("leave_out_", "", regex=False)

print("Delta ARI by left-out image and k (positive = removing this image helped):")
print(loo.pivot(index="left_out", columns="k", values="delta_ari").to_string())

print("\nDelta NMI by left-out image and k (positive = removing this image helped):")
print(loo.pivot(index="left_out", columns="k", values="delta_nmi").to_string())

print("\nRanking by mean delta_ari across k=3,4 (highest = most beneficial to remove):")
ranking = loo.groupby("left_out")[["delta_ari", "delta_nmi"]].mean().sort_values("delta_ari", ascending=False)
print(ranking.to_string())

Delta ARI by left-out image and k (positive = removing this image helped):
k                3       4
left_out                  
cor         0.0019  0.0000
gfc_sag    -0.0044 -0.0056
gfc_tra    -0.0041 -0.0107
masked_tra  0.0064  0.0027
sbj_sag     0.0027 -0.0022

Delta NMI by left-out image and k (positive = removing this image helped):
k                3       4
left_out                  
cor        -0.0100 -0.0155
gfc_sag    -0.0149  0.0126
gfc_tra    -0.0047  0.0027
masked_tra -0.0062  0.0162
sbj_sag     0.0010  0.0082

Ranking by mean delta_ari across k=3,4 (highest = most beneficial to remove):
            delta_ari  delta_nmi
left_out                        
masked_tra    0.00455    0.00500
cor           0.00095   -0.01275
sbj_sag       0.00025    0.00460
gfc_sag      -0.00500   -0.00115
gfc_tra      -0.00740   -0.00100
